In [26]:
import os
import sys
import multiprocessing


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyProBound_operator as pbo
import logomaker

In [3]:
print('#######################################################################################')
print('#### Data Wrangling with SmileSeq sequences for revisions ©Antoni Gralak_23.05.2025####')
print('#######################################################################################')

#######################################################################################
#### Data Wrangling with SmileSeq sequences for revisions ©Antoni Gralak_23.05.2025####
#######################################################################################


In [27]:
#Defining dependencies and stuff needed for the script to running properly
def _dictionary():
    return {
        'exp1': 'SmSAG01',
        'exp2': 'SmSAG02',
        'exp3': 'SmSAG03',
        'exp4': 'SmSAG04',
        'exp5': 'SmSAG05',
        'exp6': 'SmSAG06',
        'exp7': 'SmSAG07',
        'exp8': 'SmSAG08',
        'exp9': 'SmSAG09',
        'exp10': 'SmSAG10',
        'exp11': 'SmSAG11',
        'exp12': 'SmSAG12',
        'exp13': 'SmSAG13',
        'exp14': 'SmSAG14',
        'exp15': 'SmSAG15',
        'exp16': 'SmSAG16',
        'exp17': 'SmSAG17',
        'exp18': 'SmSAG18',
        'exp19': 'SmSAG19',
        'exp20': 'SmSAG20',
        'exp21': 'SmSAG21',
        'exp22': 'SmSAG22',
        'exp23': 'SmSAG23',
        'exp24': 'SmSAG24',
        '1': 'BC1',
        '2': 'BC2',
        '3': 'BC3',
        '4': 'BC4',
        '5': 'BC5',
        '6': 'BC6',
        '7': 'BC7',
        '8': 'BC8',
        '9': 'BC9',
        '10': 'BC10',
        '11': 'BC11',
        '12': 'BC12',
        'input1_fwd/rev': '20220315_input',
        'input2_fwd/rev': '20220915_input',
        'input3_fwd/rev': '20230228_input',
        'input4_fwd/rev': '20230503_input'
}
metadata_dict = _dictionary()

def assign_mBC(input_spec):
    """function accepts a character string as input_spec, so input1_fwd/rev etc. 
    Returns as first element methylated Barcode, as second the unmethylated counterpart, and the path to the data."""
    if input_spec == 'input1_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAGT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20220315_input/00_read_in_data/output/'
    elif input_spec == 'input2_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20220915_input/00_read_in_data/output/'
    elif input_spec == 'input3_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20230228_input/00_read_in_data/output/'
    elif input_spec == 'input4_fwd/rev':
        methylated_BC = "AGTA"
        unmethylated_BC = "GAAT"
        input_path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20230503_input/00_read_in_data/output/'
    
    return (methylated_BC, unmethylated_BC, input_path)

# Since I some barcodes might have mutations, these two functions will allow to identify the mBCs with
# a Hamming distance 2 and chage them to the respective BC.


def _hamming_distance(s1, s2):
    """
    Calculates the Hamming distance between two strings s1 and s2.
    """
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def _find_similar_strings(input_str, strings):
    """
    Finds strings in the list `strings` that have a Hamming distance of 1
    from the input string `input_str`.
    """
    similar_strings = []
    for s in strings:
        if _hamming_distance(input_str, s) < 2:
            similar_strings.append(True)
        else:
            similar_strings.append(False)
    return similar_strings


def barcode_correction(df, methylation_BC):
    """
    Corrects barcodes in the 'methl' column of the DataFrame by replacing 
    any string within Hamming distance <= 1 of the reference barcodes.

    Parameters:
    - df: pandas DataFrame with a 'methl' column
    - methylated_BC: reference string for methylated barcode (e.g. 'AGTA')
    - unmethylated_BC: reference string for unmethylated barcode (e.g. 'GAAT')

    Returns:
    - df: corrected DataFrame (modified in place)
    """
    mBCs = df['methl']

    similar_to_meth = _find_similar_strings(methylation_BC, mBCs)
    df.loc[similar_to_meth, 'methl'] = methylation_BC

    return df

In [4]:
# To handle eluted files, multiprocessing is available, but careful not to overload, proBound takes 4 cores by default
num_cores = 4

In [ ]:
#Read in metadata_file
metadata_path = '/data/gralak/meSMiLEseq_github/meSMiLEseq/metadata.csv'
#Due to server crash, needed to do another one
#metadata_path = '/data/gralak/meSMiLEseq_github/meSMiLEseq/metadata_delete_later.csv'
metadata = pd.read_csv(metadata_path, sep=',')

In [ ]:
#Select which data to access. Here I will only use those data sets that were accepted for initial publication
#curated_data = metadata[metadata['False'] == True].reset_index(drop=True)
curated_data = metadata[metadata['approved'] == True].reset_index(drop=True)

In [22]:
# read in data of choice and perform motif discovery separately
all_dfs = []
for row in range(len(curated_data)):
    
    exp_id = metadata_dict[curated_data['experiment'][row]]
    chip_pos = metadata_dict[str(curated_data['Chip_pos'][row])]
    TF_name = curated_data['TF'][row]
    methylated_BC, unmethylated_BC, input_path = assign_mBC(curated_data['input_library'][row])

    TF_path = f"/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/{exp_id}/00_read_in_data/output/{chip_pos}_contamination_filtered.csv"
    df = pd.read_csv(TF_path)

    df = barcode_correction(df, methylated_BC)
    df = barcode_correction(df, unmethylated_BC)

    #split based on methylated BC, for separate analysis
    methl_df = df[df['methl'] == methylated_BC].reset_index(drop=True)
    unmethl_df = df[df['methl'] == unmethylated_BC].reset_index(drop=True)

    all_dfs.append(('methylated', exp_id, chip_pos, TF_name, input_path, methl_df))
    all_dfs.append(('unmethylated', exp_id, chip_pos, TF_name, input_path, unmethl_df))

In [24]:
# Use split data by methylation to infer separate motifs.

def process_files(chunk):
    for mstat, exp, pos, TF, input, df in chunk:
    
        # output_paths
        calc_path = f'/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_separated_analysis/calc/{TF}/{exp}/{mstat}/'
        psam_path = f'/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_separated_analysis/psam/{TF}/{exp}/{mstat}/'

        ######################################################

        try:
            os.makedirs(calc_path)
        except FileExistsError:
            pass

        try:
            os.makedirs(psam_path)
        except FileExistsError:
            pass

        ######################################################

        #load correct input files
        input_path = os.path.join(input, f'{pos}_contamination_filtered.csv')
        input_df = pd.read_csv(input_path)

        input_df = barcode_correction(input_df, df['methl'][0])


        input_df = input_df[input_df['methl'] == df['methl'][0]].reset_index(drop=True)

        # Defining variables for ProBound
        #left flank
        left_flank = df[['methl', 'BC', 'lfl']].apply(lambda row: ''.join(row.values.astype(str)), axis=1)[0]
        right_flank = df['rfl'][0]
        binding_mode_flank = 5

        binding_mode_size = [6,9,12,15,24]

        #creating a df for pyProBound
        input_PB = pd.DataFrame(
                                {'header': np.repeat('input', len(input_df))
                                })
        input_PB['sequence'] = list(input_df['random24'])

            

        eluted_PB = pd.DataFrame(
                                {'header': np.repeat('eluted', len(df))
                                })
        eluted_PB['sequence'] = list(df['random24'])


        # run ProBound
        for binding_mode in binding_mode_size:

            #########################################
            # Create config for ProBound and set env#
            #########################################
            print('Create config for ProBound and set env')
            outputfile = calc_path + f'f_{TF}_bm{binding_mode}_{mstat}_output.tsv'
            count_table = pbo.build_count_table(input_PB, eluted_PB,
                                        output_filename=outputfile, gzip=False)
            
            # the default tested configuration for smile seq with three binding modes
            config = pbo.generate_SMiLE_seq_configuration(outputfile,
                                                        variable_region_length=24,
                                                        left_flank=left_flank,
                                                        right_flank=right_flank,
                                                        binding_mode_flank=binding_mode_flank, 
                                                          # this must be smaller than the left and right flank size
                                                        binding_modes=3,
                                                        binding_mode_size=binding_mode)
            basename = TF + f'_bm{binding_mode}_testmodel_{mstat}'

            config.alter_output(output_path=calc_path, 
                                base_name=basename, 
                                print_trajectory=True, 
                                # if true, generates several files in the output path, one set for each binding mode
                                # <base_name>.trajectory.component<binding mode index>-<desc of file>.csv
                                verbose=False) # flipping this switch does not seem to do very much tbh
            
            # Once you are done with the config modifications, write it to file. 
            # For ProBound, only what is written to the config file counts!
            
            config_filename = os.path.join(calc_path, f'{TF}_bm{binding_mode}_{mstat}_config.json')
            config.print_json(config_filename)
            
            # Run ProBound
            print(f'Running ProBound for {TF}, {exp}, binding size {binding_mode}, {mstat}.')
            pbo.run_probound(config_filename, 
                             full_config_file=os.path.join(calc_path, f"{mstat}_tmp.fullconfig.json"),
                             save_output=os.path.join(calc_path, f"{mstat}_tmp.optimization.out"),
                             cleanup_verbose=True)
            
            # Retrieve psam
            result_filename = pbo.get_psam(os.path.join(calc_path, f"{basename}.models.json"))
            
            # Extract psam and plot dGG matrix
            for j, psam in enumerate(result_filename):
                psam.to_csv(os.path.join(psam_path, f'{TF}_bm{binding_mode}_{mstat}_bindingmode_{str(j + 1)}.csv'))

                fig, ax = plt.subplots(1,1,figsize=[10,6])
                logo = logomaker.Logo(result_filename[j],
                                    shade_below=0.5,
                                    ax=ax,
                                    fade_below=0.5,
                                    color_scheme={'A':'#66a61e', 'C':'#7570b3','G':'#ffc809','T':'#d95f02','m':'#a6cee3'}
                                    )
                # style using Logo methods
                logo.style_spines(visible=False)
                logo.style_spines(spines=['left', 'bottom'], visible=True)
                logo.style_xticks(rotation=90, fmt='%d', anchor=0)

                # style using Axes methods
                logo.ax.set_ylabel("$-\Delta \Delta G$ (kcal/mol)", labelpad=-1)
                logo.ax.xaxis.set_ticks_position('none')
                logo.ax.xaxis.set_tick_params(pad=-1)
                #logo.ax.set_ylim([-6, 4])

                fig.suptitle(f"{TF} {mstat} bindingmode {str(j + 1)}")

                fig.savefig(os.path.join(psam_path, f'{TF}_bm{binding_mode}_{mstat}_bindingmode_{str(j + 1)}_logo.pdf'), format='pdf')
                fig.savefig(os.path.join(psam_path, f'{TF}_bm{binding_mode}_{mstat}_bindingmode_{str(j + 1)}_logo.png'), format='png')
                plt.close()
            print(f'Done with {TF}, binding size {binding_mode}, {mstat}.')

In [25]:
# Run script with n cores to create ProBound model, careful ProBound takes 4 cores
if __name__ == "__main__":
    
    #all_TF = os.listdir(raw_eluted_p)

    
    chunks = [all_dfs[i::num_cores] for i in range(num_cores)]
    
    
    with multiprocessing.Pool(processes=num_cores) as pool:
        # Call process_files function for each chunk
        pool.map(process_files, chunks)

Create config for ProBound and set env
Create config for ProBound and set env
Create config for ProBound and set env
Create config for ProBound and set env
Running ProBound for TET2_FL, SmSAG20, binding size 6, methylated.
Running ProBound for PRDM5_DBD, SmSAG18, binding size 6, methylated.
Running ProBound for PRDM5_DBD, SmSAG18, binding size 6, unmethylated.
Running ProBound for TET2_FL, SmSAG20, binding size 6, unmethylated.
Done with PRDM5_DBD, binding size 6, unmethylated.
Create config for ProBound and set env
Running ProBound for PRDM5_DBD, SmSAG18, binding size 9, unmethylated.
Done with PRDM5_DBD, binding size 6, methylated.
Create config for ProBound and set env
Running ProBound for PRDM5_DBD, SmSAG18, binding size 9, methylated.
Done with TET2_FL, binding size 6, methylated.
Create config for ProBound and set env
Running ProBound for TET2_FL, SmSAG20, binding size 9, methylated.
Done with TET2_FL, binding size 6, unmethylated.
Create config for ProBound and set env
Running P

In [37]:
calc_path = f'/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_joint_analysis/SmSAG18/04_ProBound_analysis/calc/PRDM13_DBD/binding_mode_size_12/'
basename = 'PRDM13_DBD_testmodel'
result_filename = pbo.get_psam(os.path.join(calc_path, f"{basename}.models.json"), dinucleotides=True)

In [38]:
result_filename[1]

[Empty DataFrame
 Columns: [AA, AC, AG, AT, Am, Ag, CA, CC, CG, CT, Cm, Cg, GA, GC, GG, GT, Gm, Gg, TA, TC, TG, TT, Tm, Tg, mA, mC, mG, mT, mm, mg, gA, gC, gG, gT, gm, gg]
 Index: []
 
 [0 rows x 36 columns],
 Empty DataFrame
 Columns: [AA, AC, AG, AT, Am, Ag, CA, CC, CG, CT, Cm, Cg, GA, GC, GG, GT, Gm, Gg, TA, TC, TG, TT, Tm, Tg, mA, mC, mG, mT, mm, mg, gA, gC, gG, gT, gm, gg]
 Index: []
 
 [0 rows x 36 columns],
 Empty DataFrame
 Columns: [AA, AC, AG, AT, Am, Ag, CA, CC, CG, CT, Cm, Cg, GA, GC, GG, GT, Gm, Gg, TA, TC, TG, TT, Tm, Tg, mA, mC, mG, mT, mm, mg, gA, gC, gG, gT, gm, gg]
 Index: []
 
 [0 rows x 36 columns],
 Empty DataFrame
 Columns: [AA, AC, AG, AT, Am, Ag, CA, CC, CG, CT, Cm, Cg, GA, GC, GG, GT, Gm, Gg, TA, TC, TG, TT, Tm, Tg, mA, mC, mG, mT, mm, mg, gA, gC, gG, gT, gm, gg]
 Index: []
 
 [0 rows x 36 columns]]